# 課程 12 - 使用代理備忘錄減少聊天歷史

本筆記本展示如何使用 Microsoft 代理框架管理長對話中的上下文。隨著對話增長，代幣數量增加 — 最終超過模型的上下文視窗。我們透過<strong>上下文摘要模式</strong>和持久記憶的<strong>代理備忘錄</strong>來解決此問題。

## 你將學到：
1. <strong>為什麼上下文管理很重要</strong>：理解代幣限制與上下文視窗
2. <strong>具上下文感知的代理</strong>：構建能管理自身對話上下文的代理
3. <strong>上下文摘要模式</strong>：使用工具壓縮對話歷史
4. <strong>代理備忘錄</strong>：在上下文縮減後仍能存續的持久記憶

## 前置條件：
- 配置好環境變數的 Azure OpenAI 設定
- 先前課程中對基礎代理概念的理解


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## 為什麼上下文管理很重要

每個大型語言模型都有有限的<strong>上下文視窗</strong> —— 在單次請求中能處理的最大標記數。隨著多輪對話進行：

- <strong>標記數量隨每條用戶訊息和助理回覆線性增加</strong>。
- <strong>提示標記主導成本</strong>，因為整個歷史會在每輪重新傳送。
- 最終對話會<strong>超出上下文視窗</strong>，模型要麼截斷要麼報錯。

### 上下文管理策略

| 策略 | 運作方式 | 權衡 |
|---|---|---|
| <strong>截斷</strong> | 丟棄最早訊息 | 失去早期上下文 |
| <strong>摘要</strong> | 將舊訊息濃縮成摘要 | 細節有所損失，但保留重點 |
| **草稿紙 / 外部記憶** | 將關鍵事實存放在對話外 | 需調用工具，但可避開任何縮減影響 |

在本筆記本中，我們結合<strong>摘要</strong>與<strong>草稿紙工具</strong>，讓代理能在對話歷史被濃縮時依然維持連貫性。


## 創建一個具上下文感知的代理


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## 模擬一段長對話

讓我們透過一個多輪對話來了解上下文如何累積。代理人應該在多輪中保留關鍵細節（偏好、預算、旅行日期），並展現連貫性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

注意代理如何保留先前回合的上下文 — 它知道日本、壽司、寺廟、攝影、3000 美元預算、單獨旅行以及四月的時間範圍。在短對話中這運作良好，但隨著對話增長，重新傳送完整歷史會變得昂貴。

讓我們繼續更多回合的對話，看看上下文的累積效果：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 上下文摘要模式

隨著對話成長，我們可以使用<strong>摘要工具</strong>將累積的上下文整理成精簡格式。代理人呼叫此工具以記錄關鍵偏好，這樣即使較早的訊息被刪除，重要資訊仍然被保留。

此模式是更複雜歷史縮減的基礎：
1. 代理人從對話中識別關鍵事實
2. 呼叫摘要工具以保存這些資訊
3. 因為摘要捕捉了重要內容，較舊訊息可以安全刪除

以下我們定義了一個 `summarize_preferences` 工具，代理人可以呼叫它來記錄所學到的精簡摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 摘要

在這個課程中，您學到了如何使用 Microsoft Agent Framework 管理長時間運行的代理對話中的上下文：

### 主要概念
- <strong>上下文視窗是有限的</strong> — 對話歷史中的每個代幣都會產生費用並計入限制。
- <strong>摘要工具</strong> 讓代理將累積的上下文濃縮成簡潔摘要，減少代幣使用同時保留重要信息。
- <strong>代理備忘錄</strong> 提供持久的外部記憶，能夠保存任何對話縮減後的內容。

### 您建立了什麼
- 一個 <strong>具備上下文感知能力的代理</strong>，能在多輪對話中保持連貫性
- 一個 <strong>摘要工具</strong>（`summarize_preferences`），以簡潔格式記錄用戶關鍵細節
- 一個 <strong>多輪對話示範</strong>，展示上下文保持與變化處理

### 實際應用
- <strong>客服機器人</strong>：在長時間的支援會話中記住偏好設定
- <strong>個人助理</strong>：追蹤持續進行的專案而無需重新解釋上下文
- <strong>教育輔導</strong>：維持學生多次互動中的學習進度

### 下一步
- 實現具有檔案持久性的完整備忘錄工具
- 在摘要後加入自動歷史截斷功能
- 與向量資料庫結合用於語意記憶搜尋
- 建立能夠在數天後繼續對話並保持完整上下文的代理


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
